In [ ]:
# Package imports.
import run
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# **Introduction**

This Jupyter notebook is meant to accompany the lectures on tropical cyclones for EAS 4510: Synoptic Meteorology II.

# **Tropical Cyclone Downscaling**

We will have some fun with a tropical cyclone downscaling model I developed. You can read more about it here: https://github.com/linjonathan/tropical_cyclone_risk. This model essentially generates "synthetic" tropical cyclones using the large-scale environment represented by a climate model or a reanalysis. On a high level, the downscaling model works by randomly seeding weak vortices in space and time, evolving their track using the beta-and-advection model, and simulating their intensity using a statistical-physical intensity model. One advantage of this kind of model is that one can quickly generate a large number of tropical cyclones, which is useful for a variety of applications. In this section, we will just use it to look at some of the storms generated.

First, we will use this model to downscale the ERA5 reanalysis from 2013-2023. I have already provided the input files, so all you need to do is use the below code to run the model, which will downscale the ERA5 reanalysis from 2013 to 2023. It will generate 5 tropical cyclones per year for the 10 year period from 2013-2023, for a total of around 55 tropical cyclones. It is possible to do more, but since we are limited by computational resources, we will stick to this amount. You can generate more if you like, but please be responsible with the computational power you are using on the student server.

Run the cell below, and it will take around 1 minute for this part to run. Please be patient! After it finishes running it will return the file name of the output file in <code>fn_tracks</code>.

In [ ]:
fn_tracks = run.run_model()

While you are waiting for the synthetic hurricanes to be generated, you can read the software documentation at https://github.com/linjonathan/tropical_cyclone_risk for more information on all of the output. In addition, we will look at historical values of the potential intensity, calculated using ERA5 reanalysis data. Run the next cell, which will read in the potential intensity data computed using reanalysis.

In [ ]:
fn = 'data/era5/thermo_era5_201301_202312.nc'
ds_pi = xr.open_dataset(fn)['vmax']

> **Problem 1**: Using the below cell, plot the climatological potential intensity (1979-2023 average) during August and February. Note this corresponds to summer time in the Northern and Southern Hemispheres, respectively. The code has already been provided for you. Where are potential intensity values the highest, and where are they the lowest?

In [ ]:
def plot_vmax(data_var, caption, **kwargs):
    fig = plt.figure()
    proj = ccrs.Mercator()
    fig.set_size_inches(10,5)
    ax = fig.add_axes((0.1,0.25,0.8,0.7), projection=proj)
    data_var.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(), 
                             cmap = 'Spectral_r', **kwargs)
    ax.set_global()
    ax.set_extent([0, 360, -60, 60], crs=ccrs.PlateCarree())
    ax.coastlines()
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                linewidth=2, color='gray', alpha=0.5, zorder = 0, linestyle='--')
    gl.right_labels = gl.top_labels = False
    fig.text(0.2, 0.15, caption)

plot_vmax(ds_pi.sel(time = ds_pi['time'].dt.month == 8).mean('time'),
            'Figure: Climatological potential intensity during August', vmin = 0, vmax = 120)
plot_vmax(ds_pi.sel(time = ds_pi['time'].dt.month == 2).mean('time'),
            'Figure: Climatological potential intensity during February', vmin = 0, vmax = 120)

Finally, when the synthetic track generation is finished, read and open the output dataset:

In [ ]:
ds = xr.open_dataset(fn_tracks, engine = 'netcdf4')
print(ds)

Run the next cell, which postprocesses the output of the file by defining the genesis of the tropical cyclone to be the first point at which it achieves 35 knots (18 m/s), and the lysis to be the last time point in its track where the intensity is 15 m/s. The output of this is going to be the longitude of the tropical cyclone center <code>lon_tracks</code>, the latitude of the tropical cyclone center <code>lat_tracks</code>, and the intensity of the tropical cyclone <code>vmax_tracks</code>.

In [ ]:
# Process the raw track outputs.
vmax_raw = ds['vmax_trks'].load()
lon_raw = ds['lon_trks'].load()
lat_raw = ds['lat_trks'].load()
year_tracks = ds['tc_years'].load()
month_tracks = ds['tc_month'].load()

lon_tracks = np.full(ds['lon_trks'].shape, np.nan)
lat_tracks = np.full(ds['lat_trks'].shape, np.nan)
vmax_tracks = np.full(ds['vmax_trks'].shape, np.nan)

u250 = ds['u250_trks'].load().data
v250 = ds['v250_trks'].load().data    
u850 = ds['u850_trks'].load().data
v850 = ds['v850_trks'].load().data
shear_raw = np.sqrt(np.power(u250-u850, 2) + np.power(v250-v850, 2))
shear_tracks = np.full(ds['lon_trks'].shape, np.nan)

# Here, we only consider a TC from the first point where it exceeds
# the threshold, to the point it decays to 10 m/s (after it has
# reached its peak intensity).
lon_genesis = np.full(lon_tracks.shape[0], np.nan)
lat_genesis = np.full(lon_tracks.shape[0], np.nan)
for i in range(0, lon_tracks.shape[0]):
    if len(np.argwhere(vmax_raw[i, :].data >= 15).flatten()) > 0:
        # Genesis occurs when the TC first achieves 35 knots (18 m/s).
        gen_idxs = np.argwhere(vmax_raw[i, :].data < 18).flatten()
        idx_gen = np.argwhere(vmax_raw[i, :].data >= 18).flatten()[0]
        lon_genesis[i] = lon_raw[i, idx_gen]
        lat_genesis[i] = lat_raw[i, idx_gen]

        # TC decays after it has reached 15 m/s
        decay_idxs = np.argwhere(vmax_raw[i, :].data < 15).flatten()
        idxs_lmi = np.argwhere(decay_idxs >= np.nanargmax(vmax_raw[i, :].data)).flatten()
        if len(decay_idxs) > 0 and len(idxs_lmi) > 0:
            idx_decay = decay_idxs[idxs_lmi[0]]
        else:
            idx_decay = vmax_raw.shape[1]

        nt = idx_decay - idx_gen
        vmax_tracks[i, 0:nt] = vmax_raw[i, idx_gen:idx_decay]
        lon_tracks[i, 0:nt] = lon_raw[i, idx_gen:idx_decay]
        lat_tracks[i, 0:nt] = lat_raw[i, idx_gen:idx_decay]
        shear_tracks[i, 0:nt] = shear_raw[i, idx_gen:idx_decay]

lon_tracks = xr.DataArray(lon_tracks, dims = lon_raw.dims,
                          coords = lon_raw.coords)
lat_tracks = xr.DataArray(lat_tracks, dims = lat_raw.dims,
                          coords = lat_raw.coords)
vmax_tracks = xr.DataArray(vmax_tracks, dims = vmax_raw.dims,
                           coords = vmax_raw.coords)
shear_tracks = xr.DataArray(shear_tracks, dims = vmax_raw.dims,
                            coords = vmax_raw.coords)

The variables `lon_tracks` and `lat_tracks` store the longitude and latitude (i.e. track) of each tropical cyclone. The variable `vmax_tracks` stores the intensity of the storm along each track. Finally, the variable `shear_tracks` stores the vertical wind shear along each track.

The first thing that we will check are the tropical cyclone tracks. I have provided a plotting function <code>plot_tracks</code> which will let you plot tracks nicely on a map.

In [ ]:
def plot_tracks(lon_tracks, lat_tracks):
    fig = plt.figure()
    proj = ccrs.Mercator()
    fig.set_size_inches(10,5)
    ax = fig.add_axes((0.1,0.25,0.8,0.7), projection=proj)
    for i in lon_tracks['n_trk']:
        mask = np.isfinite(lon_tracks.sel(n_trk = i))
        ax.plot(lon_tracks.sel(n_trk = i)[mask],
                lat_tracks.sel(n_trk = i)[mask],
                transform = ccrs.PlateCarree())
    ax.set_global()
    ax.set_extent([260, 350, 5, 55], crs=ccrs.PlateCarree())
    ax.coastlines()
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                linewidth=2, color='gray', alpha=0.5, zorder = 0, linestyle='--')
    gl.right_labels = gl.top_labels = False

> **Problem 2**: Using the below code, plot the tracks of the synthetic hurricanes. It will select 20 random tropical cyclones to plot. You can run it a few times to see the spread amongst the tracks. Where do the hurricanes form, and what do you observe the general motion of the hurricanes to be? In what regions over the Atlantic would you expect to have higher risk to hurricane landfall?

In [ ]:
N = 20
random_indices = np.random.choice(lon_tracks['n_trk'].size, 20, replace=False)

plot_tracks(lon_tracks.isel(n_trk = random_indices),
            lat_tracks.isel(n_trk = random_indices))

The next thing we want to do is to look at a map of where hurricane genesis occurs. This will basically tell us the spatial dependence of hurricane genesis. Note, the map will be a little noisy since we are not generating a large number of tracks. Code that counts the genesis of hurricane counts in 5-by-5 degree longitude-latitude bins has been provided for you. 

> **Problem 3**: Using <code>plot_figure</code>, plot the 2-D genesis count of the synthetic hurricanes. Where do you observe genesis to primarily occur?

In [ ]:
def plot_figure(data_var, **kwargs):
    fig = plt.figure()
    proj = ccrs.Mercator()
    fig.set_size_inches(10,5)
    ax = fig.add_axes((0.1,0.25,0.8,0.7), projection=proj)
    data_var.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(), 
                             cmap = 'hot_r', **kwargs)
    ax.set_global()
    ax.set_extent([260, 350, 5, 55], crs=ccrs.PlateCarree())
    ax.coastlines()
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                linewidth=2, color='gray', alpha=0.5, zorder = 0, linestyle='--')
    gl.right_labels = gl.top_labels = False

# Calculate the genesis density
lon_bins = np.arange(260, 351, 5)
lat_bins = np.arange(5, 56, 5)
genesis_count, lon_cbin, lat_cbin = np.histogram2d(lon_genesis, lat_genesis, bins = [lon_bins, lat_bins])

genesis_count = xr.DataArray(data = genesis_count.T, dims = ['latitude', 'longitude'],
                             coords = {'longitude': (lon_bins[1:] + lon_bins[0:-1])/2,
                                       'latitude': (lat_bins[1:] + lat_bins[0:-1])/2})

### WRITE CODE HERE ###

The month of the synthetic hurricane is also stored in the variable, <code>month_of_tc</code>. We can look at the month of the North Atlantic hurricanes to see what the seasonal cycle of hurricane activity is.

> **Problem 4**: Using the variable  <code>month_of_tc</code>, plot a histogram of the month by which the synthetic hurricanes occur. Does this align well with the climatology of tropical cyclones in the North Atlantic? When is "peak" hurricane season? Does this coincide with when you observe potential intensity to be large?

In [ ]:
month_of_tc = ds['tc_month']

### WRITE CODE HERE ###

Next, let us look at the intensity of the synthetic tropical cyclones over time. We can do this by using the variable <code>vmax_tracks</code> that was processed above.

> **Problem 5**: Using the below provided code, plot the intensities over time of the synthetic hurricanes. Again, it will select 20 random tropical cyclones to plot. You can run it a few times to see the spread amongst the tracks. Write a sentence or two on the general intensity lifecycle of the synthetic hurricanes.

In [ ]:
N = 20
random_indices = np.random.choice(lon_tracks['n_trk'].size, 20, replace=False)
s_in_day = 24 * 60 * 60

plt.plot(ds['time'] / s_in_day, vmax_tracks.isel(n_trk = random_indices).T);
plt.ylabel('Intensity (m/s)');
plt.xlabel('Time (days since genesis)');

The lifetime maximum intensity of tropical cyclones is just defined quite literally to be the maximum intensity over the entire lifetime of a tropical cyclone. Let us plot the distribution of the lifetime maximum intensity of tropical cyclones in the North Atlantic.

> **Problem 6**: Using the variable <code>vmax_tracks</code>, take the maximum over the time dimension and plot a histogram of the distribution of lifetime maximum intensity. Make sure to properly label your x- and y-axes! What do you observe about the distribution of lfietime maximum intensities?

In [ ]:
### WRITE CODE HERE ###

Finally, we want to look at some characteristics of a synthetic tropical cyclone that is modeled to become particularly strong. Run the below code until you find a track that has a peak intensity of at least 100 knots, and that (qualitatively) rapidly intensifies. Save the index of the track (<code>idx_track</code>).

In [ ]:
idx_track = np.random.choice(lon_tracks['n_trk'].size, 1, replace=False)
plt.plot(ds['time'] / s_in_day, vmax_tracks.isel(n_trk = idx_track).T*1.94384);
plt.ylabel('Intensity (knots)');
plt.xlabel('Time (days)');
print(idx_track)

> **Problem 7**: Using the below provided code, plot the vertical wind shear (<code>shear_along_track</code>) along the track for your selected storm. What do you observe the vertical wind shear to be prior to the period when the storm rapidly intensifies?

In [ ]:
shear_along_track = shear_tracks.sel(n_trk = idx_track)

### WRITE CODE HERE ###